# 02 — Investissement_Etat Diagnosis
Diagnoses the atypical negative January values (§3.2) and evaluates the monthly_diff treatment.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
from lamiaty.utils.logging import setup_logging
from lamiaty.config import load_settings
from lamiaty.data.loader import load_base_btp, COL_INVESTISSEMENT
import matplotlib.pyplot as plt
import warnings

setup_logging()
settings = load_settings("../configs", project_root="..")
df_raw = load_base_btp(settings.paths.base_btp_path)

## Raw series — negative January values visible

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(df_raw.index, df_raw[COL_INVESTISSEMENT], lw=1.3, color="#2563eb")
ax.axhline(0, color="black", lw=0.8, linestyle="--")
ax.set_title("Investissement_Etat — RAW (note large negative January values)")
ax.set_xlabel("Date")
fig.tight_layout()
fig.savefig("../docs/investissement_raw.png", dpi=150, bbox_inches="tight")

## Monthly diff treatment

In [ ]:
from lamiaty.data.corrections import fix_investissement_etat
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    corrected = fix_investissement_etat(df_raw[COL_INVESTISSEMENT], method="monthly_diff", confirmed_by=None)
    if w:
        print("⚠️ Warning:", str(w[0].message))

fig, axes = plt.subplots(2, 1, figsize=(12, 7))
axes[0].plot(df_raw.index, df_raw[COL_INVESTISSEMENT], lw=1.3, color="#dc2626")
axes[0].set_title("RAW — YTD cumulative (suspected)")
axes[1].plot(corrected.index, corrected, lw=1.3, color="#16a34a")
axes[1].axhline(0, color="black", lw=0.8, linestyle="--")
axes[1].set_title("CORRECTED — monthly_diff (first difference)")
fig.tight_layout()
fig.savefig("../docs/investissement_corrected.png", dpi=150, bbox_inches="tight")

## January values before and after

In [ ]:
import pandas as pd
jan_raw = df_raw[COL_INVESTISSEMENT][df_raw.index.month == 1]
jan_corr = corrected[corrected.index.month == 1]
comparison = pd.DataFrame({"raw": jan_raw, "monthly_diff": jan_corr})
print(comparison.to_string())
print("\n⚠️ Confirm definition with TGR/MEF before using in DFM")
print("Current status: include_in_model = false")